In [1]:
import os
import qsprpred
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs
from MolEval import MolEmb 

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, missing a depende

In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from qsprpred.data.descriptors.sets import RDKitDescs

def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    #dataset.prepareDataset(
    #feature_calculators=[MorganFP(radius=2, nBits=1024)],
    #recalculate_features=True,
    #shuffle=False
    #)
    #from qsprpred.data.descriptors.sets import RDKitDescs
    
    #rdkit_descs = RDKitDescs()
    
    #dataset.addDescriptors([rdkit_descs])
    
    #dataset.descriptorSets
    return dataset

class Dataset_creator():
    def __init__(self, model_names=['RoBERTa_ZINC'], corr_thrsh= 0.95):
        self.variance = VarianceThreshold(threshold=0.0)
        self.model_names = model_names
        self.corr_thrsh = corr_thrsh
        self.selected_indices = None
        
    def fit_transform(self, dataset):
        dataset.prepareDataset(feature_calculators=[MorganFP(radius=2, nBits=4096)], recalculate_features=True, shuffle=False)
        rdkit_descs = RDKitDescs()
        dataset.addDescriptors([rdkit_descs])
        display(dataset.X.shape)
        
        dataset.descriptorSets
        dataset_embs = self.create_embs(dataset)
        dataset_embs.index = dataset.X.index
        dataset.X = pd.concat([dataset.X, dataset_embs], axis=1)
        display(dataset.X.shape)
        
        dataset_no_var_np = self.variance.fit_transform(dataset.X)
        mask = self.variance.get_support()
        dataset_without_no_var = pd.DataFrame(
            dataset_no_var_np,
            columns=dataset.X.columns[mask],
            index=dataset.X.index)
        display(dataset_without_no_var.shape)

        
        dataset_without_high_corr = self.high_correlation(dataset_without_no_var)
        display(dataset_without_high_corr.shape)
        
        dataset.X = dataset_without_high_corr
        return dataset
        
    def transform(self, dataset):
        dataset.prepareDataset(feature_calculators=[MorganFP(radius=2, nBits=4096)], recalculate_features=True, shuffle=False)
        rdkit_descs = RDKitDescs()
        dataset.addDescriptors([rdkit_descs])
        display(dataset.X.shape)

        dataset.descriptorSets
        dataset_embs = self.create_embs(dataset)
        dataset_embs.index = dataset.X.index
        dataset.X = pd.concat([dataset.X, dataset_embs], axis=1)
        display(dataset.X.shape)

        dataset_no_var_np = self.variance.transform(dataset.X)
        mask = self.variance.get_support()
        dataset_without_no_var = pd.DataFrame(
            dataset_no_var_np,
            columns=dataset.X.columns[mask],
            index=dataset.X.index)
        display(dataset_without_no_var.shape)

        dataset_without_high_corr = dataset_without_no_var[self.selected_indices]
        display(dataset_without_high_corr.shape)

        dataset.X = dataset_without_high_corr
        return dataset

    def high_correlation(self, df: pd.DataFrame):
        corr_matrix = df.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        to_drop = [column for column in upper.columns if any(upper[column] > self.corr_thrsh)]
        self.selected_indices = df.columns.difference(to_drop)
        
        return df[self.selected_indices]

        
    def create_embs(self, dataset):
        dataset.df["SMILES"] = dataset.df["Drug"]
        final_emb = pd.DataFrame()
        for model_name in self.model_names:
            extractor = MolEmb.EmbeddingExtractor(model_name=model_name, df=dataset.df)
            new_emb, dataset.df = extractor.get_embeddings()
            display(type(new_emb))
            if final_emb.empty:
                final_emb = new_emb
            else:
                final_emb = pd.concat([final_emb, new_emb], axis=1)
        final_emb.columns = final_emb.columns.astype(str)
        display(final_emb)
        return final_emb
    


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("hERG/data/herg_train_1")

X2_all = load_datasets("hERG/data/herg_val_1")

X3_all = load_datasets("hERG/data/herg_test_1")

In [4]:
cls = Dataset_creator(['Mol2Vec', 'RoBERTa_ZINC'])

In [ ]:
X1_all = cls.fit_transform(X1_all)
X2_all = cls.transform(X2_all)
X3_all = cls.transform(X3_all)

(7856, 4306)

pandas.core.frame.DataFrame

Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
X1_all.X.to_csv("hERG/mod_data/X1.1")
X2_all.X.to_csv("hERG/mod_data/X2.1")
X3_all.X.to_csv("hERG/mod_data/X3.1")
X1_all.y.to_csv("hERG/mod_data/y1.1")
X2_all.y.to_csv("hERG/mod_data/y2.1")
X3_all.y.to_csv("hERG/mod_data/y3.1")

In [ ]:
for i in range(5, 11):
    X1_all = load_datasets(f"hERG/data/herg_train_{i}")

    X2_all = load_datasets(f"hERG/data/herg_val_{i}")

    X3_all = load_datasets(f"hERG/data/herg_test_{i}")
    cls = Dataset_creator(['Mol2Vec', 'RoBERTa_ZINC'])
    X1_all = cls.fit_transform(X1_all)
    X2_all = cls.transform(X2_all)
    X3_all = cls.transform(X3_all)

    X1_all.X.to_csv(f"hERG/mod_data/X1.{i}")
    X2_all.X.to_csv(f"hERG/mod_data/X2.{i}")
    X3_all.X.to_csv(f"hERG/mod_data/X3.{i}")
    X1_all.y.to_csv(f"hERG/mod_data/y1.{i}")
    X2_all.y.to_csv(f"hERG/mod_data/y2.{i}")
    X3_all.y.to_csv(f"hERG/mod_data/y3.{i}")

(7781, 4306)

pandas.core.frame.DataFrame

Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


pandas.core.frame.DataFrame

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,0.212876,-0.399132,-1.846029,2.869238,1.987232,1.116986,-6.785501,-0.051189,7.646624,2.011179,...,-0.258041,-0.007166,-0.713486,-0.355222,0.248249,-0.000222,0.832025,-0.109064,0.176296,-0.492081
1,-0.394226,0.520917,-2.352636,0.393088,1.582313,-1.580662,-7.656655,-1.081806,6.270067,3.489516,...,-0.277865,-0.008030,-0.216480,-0.339461,-0.058866,0.260043,0.761088,-0.575070,0.424889,0.044067
2,-1.476539,-1.981109,-2.242644,1.815164,5.709440,1.520578,-13.299513,-3.354918,7.695944,-3.535589,...,0.180058,0.016422,-0.089737,-0.249807,0.280121,-0.240548,0.899879,-0.013932,0.155572,-0.345809
3,-1.476539,-1.981109,-2.242644,1.815164,5.709440,1.520578,-13.299513,-3.354918,7.695944,-3.535589,...,-0.100282,0.210594,-0.078772,-0.463370,0.049706,-0.434389,0.869687,0.159307,-0.140322,-0.684435
4,-1.476539,-1.981109,-2.242644,1.815164,5.709440,1.520578,-13.299513,-3.354918,7.695944,-3.535589,...,-0.085757,0.253374,-0.118665,-0.458058,0.055587,-0.361078,0.843094,0.143388,-0.148290,-0.657103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7776,2.827549,-4.446218,-5.762074,11.181083,0.758236,1.425544,-13.139688,-1.437049,7.392510,3.779289,...,0.017722,-0.016625,-0.480794,-0.014142,-0.001357,-0.331914,0.494707,-0.177466,0.053791,0.079346
7777,4.631338,-5.527339,-5.030637,9.504673,-0.341492,1.346763,-14.074691,-0.665808,5.563263,3.663696,...,0.053051,0.059839,-0.402611,0.142729,-0.100703,-0.183978,0.448535,-0.305474,0.191375,-0.241570
7778,3.321135,-5.109333,-7.053854,11.402190,0.393337,2.337281,-14.455520,-2.186849,6.990888,2.845119,...,0.125685,-0.002994,-0.281138,-0.188739,-0.007633,-0.620005,0.467258,-0.237871,0.288123,-0.211845
7779,2.797021,-7.420392,-3.961010,12.614594,-0.989154,-0.488589,-14.182137,-0.573629,8.236949,4.259886,...,0.041835,0.121843,-0.045364,0.289202,-0.108492,0.160275,0.707064,0.154737,0.459228,-0.271543


...

pandas.core.frame.DataFrame

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,5.279522,-3.845104,-3.735096,8.320927,-3.505901,-1.419027,-16.305666,-1.352753,6.128940,0.124700,...,0.313759,0.087428,-0.236825,-0.334576,0.096460,-0.129412,0.897496,0.048106,-0.084576,-0.857498
1,5.279522,-3.845104,-3.735096,8.320927,-3.505901,-1.419027,-16.305666,-1.352753,6.128940,0.124700,...,0.084151,0.151190,-0.185422,-0.253309,0.070256,-0.030931,0.835972,-0.055036,-0.105877,-0.951647
2,2.768492,-2.308764,-1.417228,8.225469,-2.729634,-2.231850,-18.984846,-0.989600,9.355981,-0.979068,...,0.173596,0.398015,-0.198229,0.035552,0.147963,-0.050854,1.036273,0.136415,-0.187791,-0.826478
3,2.387123,-4.227965,-2.588701,8.796483,-2.280635,-1.419829,-16.218830,-1.696244,9.962527,-2.547624,...,0.327720,0.283097,-0.327267,0.022310,0.204418,-0.193226,1.069494,0.253996,-0.178919,-0.938253
4,3.869846,-3.580424,-2.794825,7.212237,-2.798506,-3.311748,-16.677361,0.561103,8.457273,-1.227695,...,0.064249,0.426620,0.004532,-0.155770,0.123826,0.184420,1.065646,0.081516,0.107687,-0.891857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7710,5.864903,-8.952260,-4.905296,11.305442,-1.502114,-0.655288,-13.818483,-0.491267,11.362307,4.433263,...,0.265613,0.094461,-0.360432,-0.134049,0.141988,-0.292042,0.868341,-0.297692,0.368609,-0.795828
7711,4.154066,-6.501959,-4.322742,11.684138,0.062092,0.006005,-12.962876,-0.888155,7.613721,3.349412,...,-0.000163,-0.123702,-0.206495,-0.281028,0.319712,-0.538368,0.985861,-0.404039,0.648115,-0.583253
7712,4.154066,-6.501959,-4.322742,11.684138,0.062092,0.006005,-12.962876,-0.888155,7.613721,3.349412,...,-0.006600,-0.121380,-0.183968,-0.283682,0.314276,-0.558919,0.995884,-0.410397,0.655722,-0.570527
7713,5.059038,-7.846033,-4.921978,11.720115,-1.872522,-1.623772,-14.611639,-0.620443,11.146539,5.495770,...,0.320835,0.064233,-0.292113,-0.121979,0.159408,-0.278729,0.943991,-0.321183,0.460933,-0.871470


(7715, 5374)

(7715, 5331)

(7715, 4683)

In [12]:
smiles1 = pd.concat([X1_all.getDF()["Drug"], X2_all.getDF()["Drug"], X3_all.getDF()["Drug"]]).reset_index()
X_all = pd.concat([X1_all.X, X2_all.X, X3_all.X])
y_all =  pd.concat([X1_all.y, X2_all.y, X3_all.y])
for i in range(2, 11):
    X1_all2 = load_datasets(f"hERG/data/herg_train_{i}")
    X2_all2 = load_datasets(f"hERG/data/herg_val_{i}")
    X3_all2 = load_datasets(f"hERG/data/herg_test_{i}")
    smiles2 = pd.concat([X1_all2.getDF()["Drug"], X2_all2.getDF()["Drug"], X3_all2.getDF()["Drug"]]).reset_index()
    indices_map = [list(smiles1["Drug"]).index(element) for element in smiles2["Drug"]]
    display(y_all.iloc[indices_map].Y)
    display(pd.concat([X1_all2.y, X2_all2.y, X3_all2.y]).Y)
    test1 = y_all.iloc[indices_map].Y
    test2 =  pd.concat([X1_all2.y, X2_all2.y, X3_all2.y]).Y
    for j in range(len(test1)):
        if test1[j] != test2[j]:
            print(j)
    if np.array_equal(y_all.iloc[indices_map].Y, pd.concat([X1_all2.y, X2_all2.y, X3_all2.y]).Y):
        print("good")
    else: 
        print("Critical error")
        break
    X_res = X_all.iloc[indices_map]
    X1_res = X_res.iloc[:X1_all2.getDF().shape[0]]
    X2_res = X_res.iloc[X1_all2.getDF().shape[0]:-X3_all2.getDF().shape[0]]
    X3_res = X_res.iloc[-X3_all2.getDF().shape[0]:]
    X1_res.to_csv(f"hERG/mod_data/X1.{i}")
    X2_res.to_csv(f"hERG/mod_data/X2.{i}")
    X3_res.to_csv(f"hERG/mod_data/X3.{i}")
    X1_all2.y.to_csv(f"hERG/mod_data/y1.{i}")
    X2_all2.y.to_csv(f"hERG/mod_data/y2.{i}")
    X3_all2.y.to_csv(f"hERG/mod_data/y3.{i}")
    
    


QSPRID
A2ARDataset_0000    True
A2ARDataset_0001    True
A2ARDataset_0002    True
A2ARDataset_0003    True
A2ARDataset_0004    True
                    ... 
A2ARDataset_3332    True
A2ARDataset_7852    True
A2ARDataset_7853    True
A2ARDataset_7854    True
A2ARDataset_7855    True
Name: Y, Length: 14322, dtype: bool

QSPRID
A2ARDataset_0000    True
A2ARDataset_0001    True
A2ARDataset_0002    True
A2ARDataset_0003    True
A2ARDataset_0004    True
                    ... 
A2ARDataset_3321    True
A2ARDataset_3322    True
A2ARDataset_3323    True
A2ARDataset_3324    True
A2ARDataset_3325    True
Name: Y, Length: 14322, dtype: bool

/tmp/ipykernel_27277/1241124115.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if test1[j] != test2[j]:


4978
5349
8790
9419
10146
11594
11977
11978
13979
Critical error


In [13]:
y_all.Y.sum()

8488

In [14]:
y_all.Y.count()

14322

In [16]:
smiles1.iloc[4978]

index                                                 4978
Drug     CNC(=O)C12CC1C(n1cnc3c(NC)nc(C#Cc4cccs4)nc31)C...
Name: 4978, dtype: object

In [19]:
smiles1["Drug"].nunique()

14201